In [ ]:
import pyguidos
from pyguidos import data

import rasterio
import numpy as np
import os
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap, BoundaryNorm

In [ ]:
data_dir = data.test_data_dir()

in_dir = data_dir / 'binarymap'
out_dir = Path('/home/user/tmp/output/') # <<< REPLACE with your actual desired output path
                                         #     The folder must be empty

In [ ]:
bin_map = input_dir / 'binarymap.tif'

with rasterio.open(bin_map) as src:
    array = src.read(1)

fig, ax = plt.subplots(figsize=(10, 8))

colors = ['white', 'lightgrey', 'darkgreen'] 
cmap = ListedColormap(colors)
im = ax.imshow(array, cmap=cmap, vmin=-0.5, vmax=2.5)
legend_labels = {
    0: 'Missing/NoData', 
    1: 'Background',     
    2: 'Foreground'      
}

patches = []
for val in sorted(legend_labels.keys()): 
    label = legend_labels[val]
    color = colors[val] 
    patch = mpatches.Patch(color=color, label=label)
    patches.append(patch)
legend = ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.axis('off')
plt.show()

In [ ]:
pixel_res = 100
thresholds = [200, 2000, 20000, 100000, 200000]

pyguidos.gwb_acc(in_dir, out_dir, conn_8=True, pix_res=100, thresh=thresholds)

In [ ]:
map_name = str(bin_map).split('/')[-1][:-4]
res_dir = os.path.join(out_dir, map_name + '_acc')

acc_map = os.path.join(res_dir, map_name + '_acc.tif')
acc_map_csv = os.path.join(res_dir, map_name + '_acc.csv')
acc_map_txt = os.path.join(res_dir, map_name + '_acc.txt')

In [ ]:
df = pd.read_csv(acc_map_csv, skiprows=1, header=0)
df

In [ ]:
with rasterio.open(acc_map) as src:
    array = src.read(1)

unique_values = np.unique(array)
unique_values.sort()

fig, ax = plt.subplots(figsize=(10, 8))

class_info = {
    129: {'label':'Missing/NoData', 'color': 'white'}, 
    0: {'label':'Background', 'color': 'lightgrey'},
    103: {'label':'[1 - 200]', 'color': 'black'},     
    33: {'label':'[201 - 2000]', 'color': 'red'},
    65: {'label':'[2001 - 20000]', 'color': 'yellow'},
    1: {'label':'[20001 - 100000]', 'color': 'orange'},
    9: {'label':'[100001 - 200000]', 'color': 'brown'},
    17: {'label':'[200001 -> ]', 'color': 'green'}
}

colors_list = []
labels_list = []
bounds = []
value_to_color_index = {}
for i, val in enumerate(unique_values):
    colors_list.append(class_info[val]['color'])
    labels_list.append(class_info[val]['label'])
    value_to_color_index[val] = i 
    
cmap = ListedColormap(colors_list)
boundaries = (unique_values[:-1] + unique_values[1:]) / 2
boundaries = np.insert(boundaries, 0, unique_values[0] - 0.5)
boundaries = np.append(boundaries, unique_values[-1] + 0.5)
norm = BoundaryNorm(boundaries, cmap.N)
im = ax.imshow(array, cmap=cmap, norm=norm, interpolation='nearest')

patches = []
for val in class_info.keys():
    label = class_info[val]['label']
    color = class_info[val]['color']
    patch = mpatches.Patch(color=color, label=label)
    patches.append(patch)
legend = ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.axis('off')
plt.show()

In [ ]:
with open(acc_map_txt, 'r', encoding='utf-8') as f:
    txt = f.read()
    print(txt)